In [1]:
# @title 1. Install System Dependencies & Clone Repository
import os
from IPython.display import clear_output

# Install ffmpeg (required for video processing)
!apt-get update -y && apt-get install -y ffmpeg libgl1-mesa-glx

# Clone Wan2GP
if not os.path.exists("Wan2GP"):
    !git clone https://github.com/deepbeepmeep/Wan2GP.git
    print("Wan2GP cloned successfully.")
else:
    print("Wan2GP already cloned.")

%cd Wan2GP

# Update dependencies to avoid conflicts
!pip install --upgrade pip

print("System dependencies installed.")
clear_output()

In [ ]:
# @title 2. Install Python Dependencies (Optimized for T4)
# We use the installation path recommended for RTX 20XX/T4 (SageAttention 1.0.6 + Compatible Torch)

# Install PyTorch compatible with T4/RTX 20XX series features
!pip install torch==2.6.0+cu126 torchvision==0.21.0+cu126 torchaudio==2.6.0+cu126 --index-url https://download.pytorch.org/whl/cu126

# Install Triton (needed for optimization)
!pip install -U "triton<3.3"

# Install SageAttention v1 (Best for T4 architecture)
!pip install sageattention==1.0.6

# Install remaining requirements
!pip install -r requirements.txt

print("Python dependencies installed.")
clear_output()

In [2]:
import os, shutil

BASE = "/kaggle/working/Wan2GP"
TMP  = "/kaggle/tmp/Wan2GP"
os.makedirs(TMP, exist_ok=True)

# keep only big folders on tmp
def move_or_link(name):
    src = os.path.join(BASE, name)
    dst = os.path.join(TMP, name)
    if os.path.islink(src):
        return
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst, ignore_errors=True)
        shutil.move(src, dst)
    if not os.path.exists(src):
        os.symlink(dst, src)

move_or_link("ckpts")
# move_or_link("loras")   # optional; can be big

# IMPORTANT: make outputs a real folder (not symlink)
out = os.path.join(BASE, "outputs")
if os.path.islink(out) or os.path.isfile(out):
    os.unlink(out)
os.makedirs(out, exist_ok=True)

print("outputs is_dir =", os.path.isdir(out), "is_link =", os.path.islink(out))


outputs is_dir = True is_link = False


In [ ]:
# Downgrade NumPy to fix binary incompatibility
!pip install "numpy<2"

In [6]:
# Install Triton (needed for optimization)
!pip install -U "triton<3.3"

# Install SageAttention v1 (Best for T4 architecture)
!pip install sageattention==1.0.6

In [7]:
import os

# 1. Patch wgp.py to resolve symlinks (Fixes FileExistsError)
# We replace "local_dir = targetRoot" with "local_dir = os.path.realpath(targetRoot)"
!sed -i 's/local_dir = targetRoot/local_dir = os.path.realpath(targetRoot)/g' /kaggle/working/Wan2GP/wgp.py

# 2. Ensure the temp folder exists and the symlink is valid
target_real = "/kaggle/tmp/Wan2GP/ckpts"
target_link = "/kaggle/working/Wan2GP/ckpts"

if not os.path.exists(target_real):
    os.makedirs(target_real, exist_ok=True)

# Re-create symlink if missing or broken
if os.path.islink(target_link):
    os.unlink(target_link)
elif os.path.exists(target_link):
    import shutil
    shutil.rmtree(target_link)

os.symlink(target_real, target_link)
print(f"Fixed symlink: {target_link} -> {target_real}")

# 3. Run the WebUI
# We clear the output path to ensure a fresh start
!rm -rf /kaggle/working/Wan2GP/outputs
!mkdir -p /kaggle/working/Wan2GP/outputs

PORT = 7861
print("Starting Wan2GP...")
!python /kaggle/working/Wan2GP/wgp.py --share --profile 4 --attention sage --listen --server-port {PORT}

Fixed symlink: /kaggle/working/Wan2GP/ckpts -> /kaggle/tmp/Wan2GP/ckpts
Starting Wan2GP...
2025-12-30 14:17:10.340324: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767104230.363623    1913 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767104230.370012    1913 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute '